# 01 · O laço de controle de um agente

Um agente, em sua forma essencial, é um *laço de controle* que media a interação entre um modelo
de linguagem e um conjunto de ferramentas. Este notebook isola esse laço em sua expressão mínima:
uma única ferramenta e nenhuma abstração adicional.

A distinção operacional relevante é a seguinte. Uma chamada isolada a um LLM é uma operação
*one-shot*: dado um prompt, produz-se uma resposta e o processo encerra. Um agente, ao contrário,
itera — a cada passo o modelo decide entre emitir uma resposta final ou solicitar a execução de
uma ferramenta, cujo resultado é reincorporado ao contexto antes da decisão seguinte.

Essa alternância entre raciocínio e ação, mediada por observações, corresponde ao padrão *ReAct*
(Yao et al., 2022, arXiv:2210.03629). Restringimos o exemplo a uma ferramenta (`calculate`) para
manter o foco no mecanismo do laço, sem a distração de múltiplas opções de ação.


## Dependências

In [ ]:
!pip install -q openai

## Configuração do cliente

Empregamos a biblioteca `openai` apontando para o endpoint **gratuito da NVIDIA**
(`https://integrate.api.nvidia.com/v1`), que expõe uma interface compatível com a API da OpenAI.
A célula seguinte concentra os parâmetros de acesso: endpoint, modelo e credencial.

Para obter a credencial: criar conta em https://build.nvidia.com e gerar uma API key (prefixo
`nvapi-`). No Google Colab, a chave pode ser guardada uma única vez em *Secrets* (ícone de chave na
barra lateral) com o nome `NVIDIA_API_KEY`; a célula a seguir a detecta automaticamente. Para
utilizar outro modelo, basta substituir a variável `MODEL` por outro identificador do catálogo
(https://build.nvidia.com/models) — nenhum outro ponto do código precisa mudar.


In [ ]:
import os
import json
from getpass import getpass
from datetime import datetime
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

# Parametros de backend (alterar para usar outro provedor)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


## Definição da ferramenta

Uma ferramenta é uma função ordinária. O dicionário `TOOL_FUNCTIONS` associa o identificador
exposto ao modelo à respectiva implementação.


In [ ]:
def calculate(expression: str) -> str:
    # Avalia uma expressao aritmetica em ambiente restrito (sem builtins)
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"


TOOL_FUNCTIONS = {"calculate": calculate}


## Especificação da ferramenta

O modelo não tem acesso ao código; recebe apenas a especificação em JSON abaixo — nome, descrição
e assinatura dos parâmetros —, a partir da qual decide se e como invocar a ferramenta. A descrição
textual é parte funcional da interface: é o que permite ao modelo inferir a aplicabilidade da
ferramenta.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Avalia uma expressao matematica e retorna o resultado.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expressao Python, ex.: '2 + 2' ou '847 * 0.15'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]


## O laço de controle

Cada iteração executa quatro etapas: (1) o estado corrente da conversa é submetido ao modelo;
(2) a resposta é uma mensagem final ou uma lista de invocações de ferramenta; (3) no primeiro
caso, o laço termina e a mensagem é retornada; (4) no segundo, as ferramentas são executadas e
seus resultados anexados ao histórico, reiniciando o ciclo.

A condição de parada é `not message.tool_calls`. A ausência de invocações é o sinal, emitido pelo
próprio modelo, de que dispõe de informação suficiente para responder. A lista `messages` mantém
o estado completo da interação e constitui a memória de trabalho do agente.


In [ ]:
def run_agent(task: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": "Voce e um assistente util. Use ferramentas quando precisar. Quando tiver a resposta final, responda sem chamar ferramentas."},
        {"role": "user", "content": task},
    ]

    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
        )
        message = response.choices[0].message
        messages.append(message)

        # Condicao de parada
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"  -> {name}({args})")
            fn = TOOL_FUNCTIONS.get(name)
            result = fn(**args) if fn else f"Ferramenta desconhecida: {name}"
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})


## Execução

Uma tarefa cuja resolução requer a ferramenta de cálculo:

In [ ]:
task = "Quanto e 15% de 847?"
print("Tarefa:", task, "\n")
print("Resposta:", run_agent(task))


## A condição de parada e o controle de invocação

O laço termina quando o modelo emite uma resposta sem invocações (`tool_calls` vazio). Sob
`tool_choice="auto"`, porém, essa decisão é probabilística e varia entre modelos e execuções: um
modelo pode acionar uma ferramenta mesmo quando ela é desnecessária (*over-invocation*) — uma
manifestação concreta do problema de confiabilidade em agentes.

O parâmetro `tool_choice` é o controle determinístico desse comportamento. Com `"none"`, o modelo
fica proibido de invocar ferramentas e a saída é necessariamente direta (`tool_calls` igual a
`None`); com `"auto"`, delega-se a decisão ao modelo. A célula abaixo exibe a forma determinística
— a única que garante o caso de parada imediata independentemente do modelo.


In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Em uma frase, o que e um agente de IA?"}],
    tools=TOOLS,
    tool_choice="none",   # proibe a invocacao de ferramentas
)
m = r.choices[0].message
print("tool_calls:", m.tool_calls, "(None -> saida direta)")
print(m.content)


---
Continuação: `02_varias_tools.ipynb` — composição de múltiplas ferramentas e a evolução do
histórico ao longo de uma execução.
